# DDR Chart Generator — Training Notebook

**Before running: Runtime → Change runtime type → T4 GPU**

### Assumed setup
Your Google Drive should has StepMania packs uploaded at:
```
MyDrive/ddc_project/data/raw/
  Folder.../
    Song Name/
      Song.sm
      Song.mp3
  Some Other Pack/
    ...
```
The folder names don't matter — the code searches recursively for any folder
that contains both a .sm file and an audio file (.mp3 / .ogg / .wav).

### Run order (every session)
Cells 1-3 must be re-run every session (Colab VMs are wiped).
Cells 4+ only need to run once — results are saved to Drive.

In [36]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create output folders if they don't exist yet
!mkdir -p /content/drive/MyDrive/ddc_project/data/cache
!mkdir -p /content/drive/MyDrive/ddc_project/checkpoints
!mkdir -p /content/drive/MyDrive/ddc_project/logs
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [37]:
# ── 2. Clone your project repo ──────────────────────────────────────────
# Replace with your actual GitHub repo URL
%cd /content
!git clone https://github.com/alexseungum/ieor142b_project.git ddc-chart-gen
%cd /content/ddc-chart-gen

# Symlink Drive data into the repo so code can find it at 'data/'
!ln -sf /content/drive/MyDrive/ddc_project/data data
print('Repo cloned and data symlinked.')

/content
fatal: destination path 'ddc-chart-gen' already exists and is not an empty directory.
/content/ddc-chart-gen
ln: failed to create symbolic link 'data/data': Operation not supported
Repo cloned and data symlinked.


In [38]:
!git -C /content/ddc-chart-gen pull
import importlib
import utils.data_utils
import models.model
importlib.reload(utils.data_utils)
importlib.reload(models.model)
from utils.data_utils import build_sample, parse_sm_file
from models.model import DDRTransformer, DDRLoss, generate_chart
print('Reloaded.')
!rm -f /content/drive/MyDrive/ddc_project/data/cache/*.pkl
!git -C /content/ddc-chart-gen pull
# delete the stage 0 cache so it doesn't reload the empty one
!rm -f /content/drive/MyDrive/ddc_project/data/cache/train_diff0.pkl
!rm -f /content/drive/MyDrive/ddc_project/data/cache/val_diff0.pkl

Already up to date.
Reloaded.
Already up to date.


In [39]:
# ── 3. Install dependencies ─────────────────────────────────────────────
!pip install librosa soundfile torchaudio --quiet
print('Done.')

Done.


In [40]:
# ── 4. Verify data ───────────────────────────────────────────────────────
# Confirms the code can see your uploaded packs and parse them correctly.
import sys
sys.path.insert(0, '/content/ddc-chart-gen')

from pathlib import Path
from utils.data_utils import build_sample, parse_sm_file

DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'

all_sm    = list(Path(DATA_ROOT).rglob('*.sm'))
all_audio = [f for f in Path(DATA_ROOT).rglob('*')
             if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
print(f'Found {len(all_sm)} .sm files and {len(all_audio)} audio files')

# Count how many songs have both .sm AND audio (usable for training)
usable = []
for sm in all_sm:
    audio = [f for f in sm.parent.iterdir()
             if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
    if audio:
        usable.append((sm, audio[0]))
print(f'Usable song-audio pairs: {len(usable)}')

# Parse one song as a sanity check
sm_path, audio_path = usable[0]
sm_data = parse_sm_file(str(sm_path))
print(f'\nSample song: {sm_data["title"]}')
print(f'Charts: {[(c["difficulty"], c["meter"]) for c in sm_data["charts"]]}')

sample = build_sample(str(audio_path), str(sm_path))
if sample:
    print(f'X shape: {sample["X"].shape}  (timesteps, context_window, mel_bins)')
    print(f'y shape: {sample["y"].shape}  (timesteps, 4 arrows)')
    print(f'Step density: {(sample["y"].sum(-1) > 0).mean():.1%} of timesteps have a step')
else:
    print('build_sample returned None — check the .sm file has a dance-single chart')
sm_data = parse_sm_file(str(sm_path))
for c in sm_data['charts']:
    print(f"{c['chart_type']} | {c['difficulty']} | measures: {len(c['measures'])} | first measure rows: {len(c['measures'][0]) if c['measures'] else 0}")

Found 93 .sm files and 93 audio files
Usable song-audio pairs: 93

Sample song: 064.
Charts: [('Hard', 10), ('Easy', 5), ('Medium', 8)]
X shape: (16, 15, 80)  (timesteps, context_window, mel_bins)
y shape: (16, 4)  (timesteps, 4 arrows)
Step density: 75.0% of timesteps have a step
dance-single | Hard | measures: 1 | first measure rows: 1708
dance-single | Easy | measures: 216 | first measure rows: 4
dance-single | Medium | measures: 216 | first measure rows: 4


In [ ]:
# TRAIN
DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/ddc_project/data/cache'
CKPT_DIR  = '/content/drive/MyDrive/ddc_project/checkpoints'

%cd /content/ddc-chart-gen
import os
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['CACHE_DIR'] = CACHE_DIR
os.environ['CKPT_DIR']  = CKPT_DIR

!python train.py \
    --data_root  "$DATA_ROOT" \
    --cache_dir  "$CACHE_DIR" \
    --checkpoint_dir "$CKPT_DIR" \
    --epochs_per_stage 20 \
    --batch_size 32 \
    --d_model 256 \
    --n_layers 4 \
    --curriculum_start 0 \
    --num_workers 0

/content/ddc-chart-gen
Using device: cuda
/content/ddc-chart-gen/models/model.py:142: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
Model parameters: 3,400,549

CURRICULUM STAGE 0  (difficulty <= 0)
Found 93 song directories
Processing 93 songs with 2 workers...
Built 252 song-difficulty samples
Saved base cache to /content/drive/MyDrive/ddc_project/data/cache/base_train.pkl
[train] 0 songs after difficulty filter (<=0)
[train] 0 chunks total
Found 93 song directories
Processing 93 songs with 2 workers...
Built 252 song-difficulty samples
Saved base cache to /content/drive/MyDrive/ddc_project/data/cache/base_val.pkl
[val] 0 songs after difficulty filter (<=0)
[val] 0 chunks total
  No samples at difficulty <= 0, skipping stage.

CURRICULUM STAGE 1  (difficulty <= 1)
Loading base cache from /content/drive/MyDrive/ddc_project/data/cache/base_train.pkl
[train] 73 songs af

In [ ]:
%cd /content/ddc-chart-gen
import json
import matplotlib.pyplot as plt

with open('logs/train_history.json') as f:
    history = json.load(f)

train_h = history['train']
val_h   = history['val']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
stage_colors = {0:'#3498db', 1:'#2ecc71', 2:'#f39c12', 3:'#e74c3c', 4:'#9b59b6'}
stage_names  = {0:'Beginner', 1:'Easy', 2:'Medium', 3:'Hard', 4:'Challenge'}

for metric, ax, title in [
    ('loss',       axes[0], 'Total Loss'),
    ('step_loss',  axes[1], 'Step Placement Loss'),
    ('f1',         axes[2], 'Step F1 Score'),
]:
    for stage in range(5):
        t_vals = [x[metric] for x in train_h if x['stage'] == stage]
        v_vals = [x[metric] for x in val_h   if x['stage'] == stage]
        if not t_vals:
            continue
        color = stage_colors[stage]
        ax.plot(t_vals, color=color, alpha=0.9, label=f'{stage_names[stage]} train')
        ax.plot(v_vals, color=color, alpha=0.5, linestyle='--', label=f'{stage_names[stage]} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch within stage')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle('Training curves by curriculum stage', y=1.02)
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
!mkdir -p /content/drive/MyDrive/ddc_project/logs
!cp logs/training_curves.png /content/drive/MyDrive/ddc_project/logs/
print('Saved to Drive.')

In [ ]:
%cd /content/ddc-chart-gen
import os
from pathlib import Path

# Option A: use a song already in your dataset
DATA_ROOT = '/content/drive/MyDrive/ddc_project/data/raw'
all_sm = list(Path(DATA_ROOT).rglob('*.sm'))
for sm in all_sm:
    audio = [f for f in sm.parent.iterdir()
             if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
    if audio:
        test_audio = str(audio[0])
        print(f'Using: {sm.parent.name} — {audio[0].name}')
        break

# Option B: upload your own song (see walkthrough below)
# from google.colab import files as colab_files
# uploaded = colab_files.upload()
# test_audio = list(uploaded.keys())[0]

CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
os.environ['TEST_AUDIO'] = test_audio
os.environ['CKPT_DIR']   = CKPT_DIR

!python generate.py \
    --audio      "$TEST_AUDIO" \
    --checkpoint "$CKPT_DIR/best_model.pt" \
    --difficulty 2 \
    --threshold  0.5 \
    --output     output_chart

!cp -r output_chart /content/drive/MyDrive/ddc_project/
print('Chart saved to Drive.')

In [ ]:
%cd /content/ddc-chart-gen
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/ddc-chart-gen')

from models.model import DDRTransformer, generate_chart
from generate import audio_to_model_input

CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
model_args = ckpt.get('args', {})
model = DDRTransformer(
    d_model=model_args.get('d_model', 256),
    nhead=model_args.get('nhead', 8),
    num_encoder_layers=model_args.get('n_layers', 4),
    dim_feedforward=model_args.get('d_ff', 1024),
    dropout=0.0,
)
model.load_state_dict(ckpt['model_state'])
X = audio_to_model_input(test_audio)

thresholds = np.linspace(0.1, 0.9, 17)
densities  = []
for th in thresholds:
    mask, _ = generate_chart(model, X, difficulty=2, step_threshold=float(th))
    densities.append(mask.mean())

plt.figure(figsize=(7, 4))
plt.plot(thresholds, densities, 'o-', color='#e74c3c')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='default threshold')
plt.xlabel('Step threshold')
plt.ylabel('Fraction of timesteps with a step')
plt.title('Difficulty knob: threshold vs. chart density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logs/threshold_sweep.png', dpi=150)
plt.show()
!mkdir -p /content/drive/MyDrive/ddc_project/logs
!cp logs/threshold_sweep.png /content/drive/MyDrive/ddc_project/logs/

In [ ]:
%cd /content/ddc-chart-gen
CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'

from google.colab import files
files.download('output_chart/visualizer.html')
files.download('output_chart/chart.sm')
files.download(f'{CKPT_DIR}/best_model.pt')
files.download('logs/training_curves.png')
files.download('logs/threshold_sweep.png')